# 14 · JSON parsing, schema enforcement and flattening

**Scenario:** Serialize source trips into JSON and parse them with an explicit StructType.

**Source:** Databricks-provided `samples.nyctaxi.trips` only. File formats, nested structures, replays and histories in this lab are derived from that asset. No hand-created record dataset, third-party download, pandas or generated replacement data is used.

**Prerequisites:** Run notebook 00 first. Each notebook is independent; run its cells in order. Mode: **dataframe**. 

**Learning-size sample:** `max_rows` defaults to 2,000. Sorting/limiting caps result size, not the underlying scan cost. Assertions use relationships instead of hardcoded dataset counts because sample assets may change.


## Step 1 — Load and inspect the Databricks asset

Read the built-in sample and normalize column names. A stable content fingerprint helps with replay examples; it is not a guaranteed real-world trip key. The original `samples` asset is always read-only.


In [ ]:
from pyspark.sql import functions as F, types as T, Window
dbutils.widgets.text('max_rows','2000')
MAX_ROWS=int(dbutils.widgets.get('max_rows'))
assert 100<=MAX_ROWS<=20000, 'Use 100–20000 rows for these bounded learning labs.'
SOURCE_ASSET='samples.nyctaxi.trips'
try:
    asset=(spark.read.table(SOURCE_ASSET)
        .select('tpep_pickup_datetime','tpep_dropoff_datetime','trip_distance',
                'fare_amount','pickup_zip','dropoff_zip')
        .orderBy('tpep_pickup_datetime','tpep_dropoff_datetime','pickup_zip',
                 'dropoff_zip','trip_distance','fare_amount')
        .limit(MAX_ROWS))
    assert asset.limit(1).count()==1, 'Databricks sample asset is empty.'
except Exception as exc:
    raise RuntimeError('This course requires the Databricks-provided samples.nyctaxi.trips asset. '
        'Check Unity Catalog/sample access on the attached compute. No synthetic or external fallback is used.') from exc
df=(asset.select(F.col('tpep_pickup_datetime').cast('timestamp').alias('pickup_ts'),
    F.col('tpep_dropoff_datetime').cast('timestamp').alias('dropoff_ts'),
    F.col('trip_distance').cast('double').alias('distance'),
    F.col('fare_amount').cast('double').alias('fare'),
    F.col('pickup_zip').cast('string').alias('pickup_zip'),
    F.col('dropoff_zip').cast('string').alias('dropoff_zip')))
df=df.withColumn('trip_id',F.sha2(F.to_json(F.struct(*[F.col(c) for c in df.columns]),
    options={'ignoreNullFields':'false'}),256))
df=df.withColumn('pickup_date',F.to_date('pickup_ts'))
df=df.withColumn('duration_sec',F.unix_timestamp('dropoff_ts')-F.unix_timestamp('pickup_ts'))
print('Source:',SOURCE_ASSET,'sample limit:',MAX_ROWS)
# trip_id is a content fingerprint, NOT an official source/business trip identifier.
# Identical source rows intentionally have identical fingerprints.
df.show(5,truncate=False)


## Step 2 — Create a payload from the source asset

This is a format transformation of real records, not a separately fabricated dataset.


In [ ]:
fields=['trip_id','pickup_ts','pickup_zip','fare','distance']
json_options={'ignoreNullFields':'false','timestampFormat':"yyyy-MM-dd'T'HH:mm:ss.SSSSSSXXX"}
payloads=df.select(F.to_json(F.struct(*fields),options=json_options).alias('payload'))
schema=T.StructType([T.StructField('trip_id',T.StringType()),T.StructField('pickup_ts',T.TimestampType()),
    T.StructField('pickup_zip',T.StringType()),T.StructField('fare',T.DoubleType()),T.StructField('distance',T.DoubleType())])
parsed=payloads.withColumn('record',F.from_json('payload',schema,options=json_options))
parsed.select('payload','record.trip_id','record.fare').show(3,truncate=False)


**Expected check:** Explicit JSON types match the documented payload contract.


## Step 3 — Flatten and check round-trip equality

Use null-safe set reconciliation when comparing decoded records; do not compare only row counts.


In [ ]:
flat=parsed.select('record.*')
expected=df.select(*fields)
assert flat.exceptAll(expected).limit(1).count()==0
assert expected.exceptAll(flat).limit(1).count()==0
print('JSON round trip preserved these fields')


**Expected check:** Both directional multiset differences are empty; timestamps are tested in the same session timezone.


## Practice

Inspect how from_json handles schema mismatches by intentionally parsing fare as an incompatible struct while keeping the source payload unchanged.

## Solution guide

The parsed field can become NULL under permissive parsing. Preserve payload plus a parse-error reason and use a documented contract; do not silently discard the original payload.

## Production considerations

Keep source lineage and stable business keys. Measure on representative volumes, review nulls and time semantics, and make retries idempotent. Sample-data checks demonstrate mechanics; they do not establish production performance or exactly-once external writes.
